<a href="https://colab.research.google.com/github/Kentakure/000-add-custom-functions/blob/master/Created-in-Google-Colab/Evacuation_Map_20Nagano%2621Gifu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests # zip ファイルをダウンロードするために追加
import zipfile  # zip ファイルを処理するために追加
import io       # メモリ内 zip ファイル処理用に追加
import tempfile # 一時ディレクトリを作成するために追加
import shutil   # 一時ディレクトリをクリーンアップするために追加
import os       # パス操作用に追加
import folium # マップ操作用に追加
from folium import plugins
import geopandas as gpd # GeoDataFrame 操作用に追加
import pandas as pd # DataFrame 操作用に追加
from geopy.geocoders import Nominatim #ジオコーディング用に追加
from datetime import datetime
from google.colab import files

address_input = input("地図の中央に配置する住所を英語またはローマ字で入力してください (例:「Tokyo Station」またはデフォルトの場合は空白のままにします): ")

# geopy を使用して住所入力から座標を取得します
if address_input:
    geolocator = Nominatim(user_agent="geoapi_colab_map")
    try:
        location = geolocator.geocode(address_input)
        if location:
            center = [location.latitude, location.longitude]
            print(f"地図の中心: {location.address} ({center[0]}, {center[1]}) ")
        else:
            print(f"デフォルトの中心を使用しているため、'{address_input}'の座標が見つかりませんでした。")
    except Exception as e:
        print(f"ジオコーディングエラー '{address_input}': {e} デフォルトの中心を使用。")

# 'center' が定義されていない場合 (たとえば、ユーザーが入力セルをスキップした場合)、堅牢性のためにデフォルトが使用されます。
if 'center' not in locals():
    print("「警告: 'center' 変数が見つかりません。デフォルトの座標を使用します。」")
    center = [35.841395, 137.4732] #https://geoshape.ex.nii.ac.jp/ka/resource/20/20429001002.html

# OpenStreetMap
fmap1 = folium.Map(location=center,
                   zoom_start = 10,
                   #zoom_control=False, # ズームボタンを非表示
                   scrollWheelZoom=False, # マウスホイールでのズームを無効化
                   doubleClickZoom=False, # ダブルクリックでのズームを無効化
                   #dragging=False # ドラッグでの移動を無効化
)

# 値「1」を「該当」に置き換え、「0」または空の値を「非該当」に置き換えるヘルパー関数
def _replace_categorical_values(gdf, columns):
    for col in columns:
        if col in gdf.columns:
            # 最初に数値に変換し、エラーを NaN に強制してから、NaN を 0 で埋めます
            # これは空の文字列、数値以外の値を「非該当」として処理します
            gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0).astype(int)
            gdf[col] = gdf[col].map({1: '該当', 0: '×非該当×'})
            # 変換後も値が 0 または 1 にならなかった場合は、残った NaN を埋めます
            gdf[col] = gdf[col].fillna('×非該当×')
    return gdf

# CSV を GeoDataFrame に変換するヘルパー関数
def csv_to_gdf(df_in, suffix_key_for_log):
    df = df_in.copy() # Work on a copy
    lat_cols = ['緯度', 'latitude', 'lat', 'Y']
    lon_cols = ['経度', 'longitude', 'lon', 'X']

    found_lat_col = None
    found_lon_col = None

    # 大文字と小文字を区別せずに一致する列を検索します
    for col in df.columns:
        if any(lc.lower() == col.lower() for lc in lat_cols):
            found_lat_col = col
        if any(lc.lower() == col.lower() for lc in lon_cols):
            found_lon_col = col
        if found_lat_col and found_lon_col:
            break

    if found_lat_col and found_lon_col:
        # 緯度/経度が欠落しているか無効な行を除外します
        df = df.dropna(subset=[found_lat_col, found_lon_col])
        # 座標が数値であることを確認する
        df[found_lat_col] = pd.to_numeric(df[found_lat_col], errors='coerce')
        df[found_lon_col] = pd.to_numeric(df[found_lon_col], errors='coerce')
        df = df.dropna(subset=[found_lat_col, found_lon_col])

        if not df.empty:
            geometry = gpd.points_from_xy(df[found_lon_col], df[found_lat_col])
            gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
            print(f"CSV から正常にロードされ、{suffix_key_for_log}の GeoDataFrame に変換されました。")
            return gdf
        else:
            print(f"CSV は見つかりましたが、{suffix_key_for_log}をクリーニングした後、有効な座標データがありませんでした。")
            return None
    else:
        print(f"CSV は見つかりましたが、{suffix_key_for_log}の緯度({lat_cols})列と経度({lon_cols})列を識別できませんでした。")
        return None

# 圧縮ファイルから空間データファイルを検索し、GeoDataFrameとしてロードするヘルパー関数
def _find_and_load_spatial_data(temp_dir, suffix_key_for_log, csv_to_gdf_func):
    gdf = None
    found_file_path = None

    # 優先順位: GeoJSON, Shapefile, CSV
    # 各形式でファイルを検索します。サブディレクトリ内も検索します。
    for root, dirs, files in os.walk(temp_dir):
        for file in files:
            if file.lower().endswith('.geojson'):
                found_file_path = os.path.join(root, file)
                try:
                    gdf = gpd.read_file(found_file_path)
                    print(f"読み込まれたGeoJSON: {found_file_path}")
                    return gdf
                except Exception as e:
                    print(f"GeoJSON '{found_file_path}'の読み取りエラー: {e}")
                    found_file_path = None # エラーが発生した場合はリセットして次の検索に進む

    if gdf is None: # GeoJSONが見つからなかった場合、Shapefileを検索
        for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.shp'):
                    found_file_path = os.path.join(root, file)
                    try:
                        gdf = gpd.read_file(found_file_path)
                        print(f"読み込まれたシェープファイル: {found_file_path}")
                        return gdf
                    except Exception as e:
                        print(f"シェープファイル '{found_file_path}'の読み取りエラー: {e}")
                        found_file_path = None # エラーが発生した場合はリセットして次の検索に進む

    if gdf is None: # Shapefileも見つからなかった場合、CSVを検索
        for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.csv'):
                    found_file_path = os.path.join(root, file)
                    try:
                        df = pd.read_csv(found_file_path)
                        gdf = csv_to_gdf_func(df, suffix_key_for_log)
                        if gdf is not None:
                            print(f"読み込まれたCSV: {found_file_path} (GeoDataFrameに変換済み)")
                        return gdf
                    except Exception as e:
                        print(f"CSV '{found_file_path}'の読み取りエラー: {e}")
                        found_file_path = None # エラーが発生した場合はリセットして次の検索に進む

    if gdf is None: #いずれのファイルも見つからなかった場合
        print(f"一時ディレクトリ '{temp_dir}'にGeoJSON、Shapefile (.shp)、または CSV (.csv) が見つかりません。")
    return gdf

# 国土地理院 GeoJSON/Shapefile/CSV データをダウンロードして処理するヘルパー関数
def _get_gsi_data(prefix, suffix_key, info, base_url, csv_to_gdf_func):
    zip_filename = f'{prefix}{suffix_key}.zip'
    zip_url = f'{base_url}{zip_filename}'
    gdf = None

    print(f"ダウンロードと処理中: {zip_url}")

    try:
        response = requests.get(zip_url)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            temp_dir = None
            try:
                temp_dir = tempfile.mkdtemp()
                zf.extractall(temp_dir)

                # 新しいヘルパー関数を呼び出して空間データを検索し、ロードします
                gdf = _find_and_load_spatial_data(temp_dir, suffix_key, csv_to_gdf_func)

            finally:
                if temp_dir and os.path.exists(temp_dir):
                    shutil.rmtree(temp_dir)

    except requests.exceptions.RequestException as e:
        print(f"{zip_url}のダウンロード中にエラーが発生しました: {e}")
    except zipfile.BadZipFile as e:
        print(f"zipファイル{zip_filename}を開くときにエラーが発生しました: {e}")
    except Exception as e:
        print(f"{zip_filename}の処理中に予期しないエラーが発生しました: {e}")
    return gdf

# 国土数値情報 GeoJSON/Shapefile/CSV データをダウンロードして処理するためのヘルパー関数
def _get_ksj_data(prefix, base_url, csv_to_gdf_func):
    zip_filename = f'{prefix}_GML.zip'
    zip_url = f'{base_url}/{zip_filename}'
    gdf = None

    print(f"ダウンロードと処理中: {zip_url}")

    try:
        response = requests.get(zip_url)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            temp_dir = None
            try:
                temp_dir = tempfile.mkdtemp()
                zf.extractall(temp_dir)

                # 新しいヘルパー関数を呼び出して空間データを検索し、ロードします
                gdf = _find_and_load_spatial_data(temp_dir, prefix, csv_to_gdf_func)

            finally:
                if temp_dir and os.path.exists(temp_dir):
                    shutil.rmtree(temp_dir)

    except requests.exceptions.RequestException as e:
        print(f"{zip_url}のダウンロード中にエラーが発生しました: {e}")
    except zipfile.BadZipFile as e:
        print(f"zipファイル{zip_filename}を開くときにエラーが発生しました: {e}")
    except Exception as e:
        print(f"{zip_filename}の処理中に予期しないエラーが発生しました: {e}")
    return gdf

# 避難所等データのジオデータフレームを作成する
# 出典：国土地理院 避難所等データダウンロードサイト
# https://hinanmap.gsi.go.jp/hinanjocp/hinanbasho/koukaidate.html

base_url = 'https://japanhazardmap.github.io/DATA/GIS/MLITGSI/DesignatedEvacuationSitesandShelters/'
prefixes = ['20000','21000']
suffixes = {'_1': {'name': '指定避難所', 'fields': ['共通ID','施設・場所名','住所','指定緊急避難場所との住所同一','その他市町村長が必要と認める事項','受入対象者','備考']},
            '_2': {'name': '指定緊急避難場所', 'fields': ['共通ID','施設・場所名','住所','洪水','崖崩れ、土石流及び地滑り','高潮','地震','津波','大規模な火事','内水氾濫','火山現象','指定避難所との住所同一','備考']}}

shelter_gdfs_list = []
evac_site_gdfs_list = []

for prefix in prefixes:
    for suffix_key, info in suffixes.items():
        gdf = _get_gsi_data(prefix, suffix_key, info, base_url, csv_to_gdf)
        if gdf is not None:
            if suffix_key == '_1':
                shelter_gdfs_list.append(gdf)
            elif suffix_key == '_2':
                evac_site_gdfs_list.append(gdf)
        else:
            print(f"{prefix}{suffix_key}の空間データをロードできませんでした。")

# タイプごとに GeoDataFrame を連結する
if shelter_gdfs_list:
    combined_shelters_gdf = gpd.pd.concat(shelter_gdfs_list, ignore_index=True)
   # combined_shelters_gdf の置換を適用します
    cols_to_replace_shelters = ['指定緊急避難場所との住所同一']
    combined_shelters_gdf = _replace_categorical_values(combined_shelters_gdf, cols_to_replace_shelters)

    folium.GeoJson(
        combined_shelters_gdf,
        attribution='&copy; <a href="https://hinanmap.gsi.go.jp/hinanjocp/hinanbasho/koukaidate.html" target="_blank" rel="noopener">国土地理院</a> ',
        name=suffixes['_1']['name'], # Use the combined name
        tooltip=folium.GeoJsonTooltip(fields=suffixes['_1']['fields']),
        popup=folium.GeoJsonPopup(fields=suffixes['_1']['fields']),
        zoom_on_click=True,
    ).add_to(fmap1)

if evac_site_gdfs_list:
    combined_evac_sites_gdf = gpd.pd.concat(evac_site_gdfs_list, ignore_index=True)
    # combined_evac_sites_gdf の置換を適用します
    cols_to_replace_evac_sites = ['洪水', '崖崩れ、土石流及び地滑り', '高潮', '地震', '津波', '大規模な火事', '内水氾濫', '火山現象', '指定避難所との住所同一']
    combined_evac_sites_gdf = _replace_categorical_values(combined_evac_sites_gdf, cols_to_replace_evac_sites)

    folium.GeoJson(
        combined_evac_sites_gdf,
        attribution='&copy; <a href="https://hinanmap.gsi.go.jp/hinanjocp/hinanbasho/koukaidate.html" target="_blank" rel="noopener">国土地理院</a> ',
        name=suffixes['_2']['name'], # Use the combined name
        tooltip=folium.GeoJsonTooltip(fields=suffixes['_2']['fields']),
        popup=folium.GeoJsonPopup(fields=suffixes['_2']['fields']),
        zoom_on_click=True,
    ).add_to(fmap1)

#避難施設データ2012年度（平成24年度）版(.shp)の読み込み
#出典：国土数値情報 | 避難施設データ
#https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P20.html
base_url2 = 'https://japanhazardmap.github.io/DATA/GIS/MLITKSJ/P20Zips'
prefixes2 = ['P20-12_20','P20-12_21']
shelter_2012_gdfs_list = []

for prefix in prefixes2:
    gdf = _get_ksj_data(prefix, base_url2, csv_to_gdf)
    if gdf is not None:
        shelter_2012_gdfs_list.append(gdf)
    else:
        print(f"{prefix}の空間データをロードできませんでした。")

# 避難施設データ2012年度の GeoDataFrame を連結する
if shelter_2012_gdfs_list:
    combined_shelters_2012_gdf = gpd.pd.concat(shelter_2012_gdfs_list, ignore_index=True)

    # カテゴリカル置換を適用する前に列の名前を変更します
    combined_shelters_2012_gdf = combined_shelters_2012_gdf.rename(columns={
        'P20_001':'行政区域コード','P20_002':'名称','P20_003':'住所','P20_004':'施設の種類',
        'P20_005':'収容人数','P20_006':'施設規模(m2)','P20_007':'地震災害','P20_008':'津波災害',
        'P20_009':'水害','P20_010':'火山災害','P20_011':'災害分類その他','P20_012':'災害分類指定なし'
    })

    # combined_shelters_2012_gdf の置換を適用します
    cols_to_replace_shelters_2012 = ['地震災害', '津波災害', '水害', '火山災害', '災害分類その他', '災害分類指定なし']
    combined_shelters_2012_gdf = _replace_categorical_values(combined_shelters_2012_gdf, cols_to_replace_shelters_2012)

    # ツールチップとポップアップのフィールドを定義し、「ジオメトリ」が含まれないようにします。
    # この段階で存在する関連する属性列を使用します (例: P20_001、P20_003)。

    # 名前変更後、関連する列が表示されます
    display_fields_for_map = ['行政区域コード','名称', '住所', '施設の種類','収容人数','施設規模(m2)','地震災害', '津波災害', '水害', '火山災害', '災害分類その他', '災害分類指定なし']

    # GeoDataFrame に実際に存在するフィールドのみを含むようにフィルターします。
    tooltip_fields = [field for field in display_fields_for_map if field in combined_shelters_2012_gdf.columns]
    popup_fields = [field for field in display_fields_for_map if field in combined_shelters_2012_gdf.columns]

    # 特定の説明フィールドが見つからない場合は、最初のいくつかの利用可能な非ジオメトリ列をフォールバックとして使用します。
    if not tooltip_fields:
        non_geometry_cols = [col for col in combined_shelters_2012_gdf.columns if col != combined_shelters_2012_gdf.geometry.name]
        tooltip_fields = non_geometry_cols[:3] # Take up to 3 columns as a generic fallback
        popup_fields = non_geometry_cols[:3] # Same for popup

    # 一意のフィールドを確保します (ただし、この場合は直接割り当てを使用していますが、それほど重要ではありません)
    tooltip_fields = list(dict.fromkeys(tooltip_fields))
    popup_fields = list(dict.fromkeys(popup_fields))

    # 結合された避難施設データ2012年度をマップに追加します
    folium.GeoJson(
        combined_shelters_2012_gdf,
        attribution='&copy; <a href="https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P20.html" target="_blank" rel="noopener">国土数値情報</a> ',
        name='避難施設データ 2012年度（平成24年度）版',
        tooltip=folium.GeoJsonTooltip(fields=tooltip_fields),
        popup=folium.GeoJsonPopup(fields=popup_fields),
        zoom_on_click=True,
        show=False, # 最初はレイヤーを非表示にします
        marker=folium.CircleMarker(radius=3, fill=True, color='blue', fill_color='blue', fill_opacity=0.6)
    ).add_to(fmap1)

else:
    print("'combined_shelters_2012_gdf'は作成されませんでした。ダウンロード/処理中にエラーがないか確認してください。")
#指定避難所の合計
if 'combined_shelters_gdf' in locals():
    num_shelters = len(combined_shelters_gdf)
    print(f"指定避難所(combined_shelters_gdf)の総数は、{num_shelters}箇所。")
else:
    print("'combined_shelters_gdf'は存在しません。最初に前のセルを実行してください。")
#指定緊急避難場所の合計
if 'combined_evac_sites_gdf' in locals():
    num_evac_sites = len(combined_evac_sites_gdf)
    print(f"指定緊急避難場所(combined_evac_sites_gdf)の総数は、{num_evac_sites}箇所。")
else:
    print("'combined_evac_sites_gdf'は存在しません。最初に前のセルを実行してください。")
#避難施設 2012年度（平成24年度）版の合計
if 'combined_shelters_2012_gdf' in locals():
    num_shelters_2012 = len(combined_shelters_2012_gdf)
    print(f"避難施設2012年度版(combined_shelters_2012_gdf)の総数は、{num_shelters_2012}箇所。")
else:
    print("'combined_shelters_2012_gdf'は存在しません。最初に前のセルを実行してください。")
#各GDFの中身
display(combined_shelters_gdf)
display(combined_evac_sites_gdf)
display(combined_shelters_2012_gdf)

# ベースマップ
base_maps = [
    ('https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png', '地理院地図', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('https://cyberjapandata.gsi.go.jp/xyz/english/{z}/{x}/{y}.png', 'English', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('http://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 'Esri Satellite', 'Esri'),
    ('https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.png', '全国最新写真（シームレス）', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> ')
]
for tiles, name, attr in base_maps:
    folium.raster_layers.TileLayer(
        tiles=tiles,
        fmt='image/png' if 'gsi' in tiles else None, # 地理院地図は PNG であり、Esri は明示的ではありませんが、通常はこれを処理します
        attr=attr,
        name=name,
        overlay=False,
        control=True
    ).add_to(fmap1)

# Layer of Disasters
# 出典：ハザードマップポータルサイト
# https://disaportal.gsi.go.jp/hazardmap/copyright/opendata.html
disaster_layers = {
    '01_flood_l2_shinsuishin_kuni_data': '洪水浸水想定区域（想定最大規模）',
    '01_flood_l2_keizoku_data': '浸水継続時間（想定最大規模）',
    '01_flood_l2_kaokutoukai_hanran_data': '家屋倒壊等氾濫想定区域（氾濫流）',
    '01_flood_l2_kagan_data': '家屋倒壊等氾濫想定区域（河岸侵食）',
    '02_naisui_data': '内水（雨水出水）浸水想定区域',
    '03_hightide_l2_shinsuishin_data': '高潮浸水想定区域',
    '04_tsunami_newlegend_data': '津波浸水想定',
    '05_dosekiryukeikaikuiki': '土砂災害警戒区域（土石流）',
    '05_kyukeishakeikaikuiki': '土砂災害警戒区域（急傾斜地の崩壊）',
    '05_jisuberikeikaikuiki': '土砂災害警戒区域（地すべり）',
    '05_nadarekikenkasyo': '雪崩危険箇所'
}
for url_part, name in disaster_layers.items():
    folium.raster_layers.TileLayer(
        tiles=f'https://disaportal.gsi.go.jp/raster/{url_part}/{{z}}/{{x}}/{{y}}.png',
        fmt='image/png',
        attr='&copy; <a href="https://disaportal.gsi.go.jp/hazardmap/copyright/opendata.html" target="_blank" rel="noopener">ハザードマップポータルサイト</a> ',
        name = name,
        tms=False,
        overlay=True,
        control=True,
        opacity=0.7
    ).add_to(fmap1)

#レイヤーコントロール
folium.LayerControl().add_to(fmap1)

#全画面表示
folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
    ).add_to(fmap1)

#地図の中心を現在地に変更するボタン
plugins.LocateControl(
    auto_start=False,      # 読み込み時に自動で現在地を取得するかどうか
    flyToNextView=True,    # 現在地へスムーズに移動
    strings={"title": "現在地を表示"} # ボタンのホバーテキスト
).add_to(fmap1)

#出力された地図fmap1をHTMLファイルに保存する
filename = f"evacuation_map_{datetime.now().strftime("%Y-%m-%d_%H-%M")}.html"
fmap1.save(filename)
files.download(filename)



地図の中央に配置する住所を英語またはローマ字で入力してください (例:「Tokyo Station」またはデフォルトの場合は空白のままにします): mt.ontake
地図の中心: 御嶽山, 王滝村, 木曽郡, 長野県, 日本 (35.8929915, 137.4804059) 
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITGSI/DesignatedEvacuationSitesandShelters/20000_1.zip
読み込まれたGeoJSON: /tmp/tmp9gooj4v2/20000_1.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITGSI/DesignatedEvacuationSitesandShelters/20000_2.zip
読み込まれたGeoJSON: /tmp/tmpaenhcx1y/20000_2.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITGSI/DesignatedEvacuationSitesandShelters/21000_1.zip
読み込まれたGeoJSON: /tmp/tmpljm2m2cg/21000_1.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITGSI/DesignatedEvacuationSitesandShelters/21000_2.zip
読み込まれたGeoJSON: /tmp/tmpgkqjux_c/21000_2.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITKSJ/P20Zips/P20-12_20_GML.zip
読み込まれたシェープファイル: /tmp/tmp9schd67c/P20-12_20.shp
ダウンロードと処理中: https://japanhazardmap.github.io/DATA/GIS/MLITKSJ/P20Zips/P20-12_21_GML.zip


,NO,共通ID,施設・場所名,住所,指定緊急避難場所との住所同一,その他市町村長が必要と認める事項,受入対象者,備考,geometry
0,1,E2040700001111,阿智第一小学校,長野県阿智村駒場143,×非該当×,None,None,,POINT (137.74654 35.44766)
1,2,E2040700003111,阿智村中央公民館,長野県阿智村駒場468-1,該当,None,None,,POINT (137.74802 35.44413)
2,3,E2040700004111,阿智中学校,長野県阿智村伍和173,該当,None,None,,POINT (137.7456 35.43868)
3,4,E2040700005111,阿智第二小学校,長野県阿智村伍和4500,該当,None,None,,POINT (137.74556 35.42578)
4,5,E2040700006111,伍和公民館,長野県阿智村伍和4555,該当,None,None,,POINT (137.74407 35.42278)
...,...,...,...,...,...,...,...,...,...
5317,1954,E2138200028111,輪之内町ふれあいセンター,岐阜県輪之内町中郷新田2201-1,該当,None,None,,POINT (136.62935 35.27845)
5318,1955,E2138200029121,特別養護老人ホームハピネスビラ,岐阜県輪之内町中郷新田2408,×非該当×,None,要配慮者,,POINT (136.63338 35.28356)
5319,1956,E2138200031111,福束小学校,岐阜県輪之内町南波76,該当,None,None,,POINT (136.63121 35.30368)
5320,1957,E2138200032111,福束小学校体育館,岐阜県輪之内町南波77,×非該当×,None,None,,POINT (136.63216 35.30407)


,NO,共通ID,施設・場所名,住所,洪水,崖崩れ、土石流及び地滑り,高潮,地震,津波,大規模な火事,内水氾濫,火山現象,指定避難所との住所同一,備考,geometry
0,1,E2021700001201,馬坂特農館,群馬県南牧村842-1,該当,該当,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,×非該当×,,POINT (138.63566 36.17078)
1,2,E2040700002201,阿智第一小学校校庭,長野県阿智村駒場143-1,該当,×非該当×,×非該当×,該当,×非該当×,該当,×非該当×,×非該当×,×非該当×,,POINT (137.74659 35.44726)
2,3,E2040700003201,阿智村北側駐車場,長野県阿智村駒場468-1,該当,該当,×非該当×,該当,×非該当×,該当,×非該当×,×非該当×,該当,,POINT (137.74762 35.44416)
3,4,E2040700004201,阿智中学校校庭,長野県阿智村伍和173,該当,×非該当×,×非該当×,該当,×非該当×,該当,×非該当×,×非該当×,該当,,POINT (137.74671 35.43842)
4,5,E2040700005201,阿智第二小学校校庭,長野県阿智村伍和4500,該当,該当,×非該当×,該当,×非該当×,該当,×非該当×,×非該当×,該当,,POINT (137.74648 35.42577)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7130,3166,E2138200041201,福束こども園,岐阜県輪之内町里983-1,該当,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,,POINT (136.63268 35.30392)
7131,3167,E2138200042201,白鬚神社（楡俣）,岐阜県輪之内町楡俣1698-1,×非該当×,×非該当×,×非該当×,該当,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,,POINT (136.66286 35.30168)
7132,3168,E2138200043201,済美館,岐阜県輪之内町楡俣2261-4,×非該当×,×非該当×,×非該当×,該当,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,,POINT (136.65646 35.30704)
7133,3169,E2138200044201,長良川堤,岐阜県輪之内町楡俣3868-4,該当,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,,POINT (136.66472 35.29926)


,行政区域コード,名称,住所,施設の種類,収容人数,施設規模(m2),地震災害,津波災害,水害,火山災害,災害分類その他,災害分類指定なし,レベル,備考,緯度,経度,NO,geometry
0,20201,（仮称）篠ノ井中央公園,長野県長野市篠ノ井会723-1,避難地,-1,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,36.573820,138.152025,1,POINT (138.15203 36.57382)
1,20201,ＮＴＴ総合運動公園,長野県長野市青木島町綱島,避難地,-1,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,36.618558,138.189873,2,POINT (138.18987 36.61856)
2,20201,アクアパル千曲,長野県長野市真島町川合1060-1,避難地,-1,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,36.610848,138.221430,3,POINT (138.22143 36.61085)
3,20201,アゼィリア飯綱,長野県長野市上ヶ屋2471-79,避難地、避難施設,-1,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,36.701510,138.135902,4,POINT (138.1359 36.70151)
4,20201,オリンピック記念アリーナ,長野県長野市北長池195,避難地,-1,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,1,None,36.640534,138.240273,5,POINT (138.24027 36.64053)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8446,21604,飯島集落センター,岐阜県大野郡白川村大字飯島864番地,避難所,30,177,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,1,None,36.276682,136.900500,4070,POINT (136.9005 36.27668)
8447,21604,平瀬保育園,岐阜県大野郡白川村大字平瀬126番地の10,避難所,40,77,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,1,None,36.170027,136.907297,4071,POINT (136.9073 36.17003)
8448,21604,本覚寺,岐阜県大野郡白川村大字荻町385番地,避難所,30,57,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,36.257483,136.905361,4072,POINT (136.90536 36.25748)
8449,21604,明善寺,岐阜県大野郡白川村大字荻町679番地,避難所,30,-1,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,1,None,36.255791,136.906618,4073,POINT (136.90662 36.25579)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#完成した地図fmap1を表示させる
fmap1